GITHUB URL:
https://github.com/KhadijaAR29/Static-and-Dynamic-Scraping-Datasets.git


QUESTION 01

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

BASE_URL = "https://sandbox.oxylabs.io/products"

In [3]:
session = requests.Session()

response = session.get(BASE_URL)
print(response.status_code)
print(len(response.text))

soup = BeautifulSoup(response.text, "html.parser")
print(soup.title.text)

200
137399
E-commerce	| Oxylabs Scraping Sandbox


In [4]:
DOMAIN = "https://sandbox.oxylabs.io"

def parse_listing_page(html, page_num):
    soup = BeautifulSoup(html, "html.parser")
    cards = soup.select(".product-card")
    products = []
    for card in cards:
        title_tag = card.select_one("a")
        price_tag = card.select_one(".price-wrapper")

        name = title_tag.select_one("h4").text.strip() if title_tag else None
        url = DOMAIN + title_tag["href"] if title_tag else None
        price = price_tag.text.strip() if price_tag else None

        products.append({
            "name": name,
            "price": price,
            "detail_url": url,
            "listing_page": page_num
        })
    return products

products = parse_listing_page(response.text, 1)
products[:3]

[{'name': 'The Legend of Zelda: Ocarina of Time',
  'price': '91,99 €',
  'detail_url': 'https://sandbox.oxylabs.io/products/1',
  'listing_page': 1},
 {'name': 'Super Mario Galaxy',
  'price': '91,99 €',
  'detail_url': 'https://sandbox.oxylabs.io/products/2',
  'listing_page': 1},
 {'name': 'Super Mario Galaxy 2',
  'price': '91,99 €',
  'detail_url': 'https://sandbox.oxylabs.io/products/3',
  'listing_page': 1}]

In [6]:
# Test A: request page 2 using the SAME session
resp_with_session = session.get(BASE_URL, params={"page": 2})
soup_a = BeautifulSoup(resp_with_session.text, "html.parser")
first_product_a = soup_a.select_one(".product-card h4")
print("With session:", first_product_a.text if first_product_a else "NOT FOUND")

# Test B: request page 2 with a completely FRESH, independent request
resp_fresh = requests.get(BASE_URL, params={"page": 2})
soup_b = BeautifulSoup(resp_fresh.text, "html.parser")
first_product_b = soup_b.select_one(".product-card h4")
print("Fresh request:", first_product_b.text if first_product_b else "NOT FOUND")

print("With session -> status:", resp_with_session.status_code, "| final URL:", resp_with_session.url, "| redirected:", len(resp_with_session.history) > 0)
print("Fresh        -> status:", resp_fresh.status_code, "| final URL:", resp_fresh.url, "| redirected:", len(resp_fresh.history) > 0)

print("Session cookies:", session.cookies.get_dict())

With session: Resident Evil Code: Veronica
Fresh request: Resident Evil Code: Veronica
With session -> status: 200 | final URL: https://sandbox.oxylabs.io/products?page=2 | redirected: False
Fresh        -> status: 200 | final URL: https://sandbox.oxylabs.io/products?page=2 | redirected: False
Session cookies: {}


In [4]:
import re
import requests
from bs4 import BeautifulSoup
import time

BASE_URL = "https://sandbox.oxylabs.io/products"
session = requests.Session()
DOMAIN = "https://sandbox.oxylabs.io"

def parse_listing_page(html, page_num):
    soup = BeautifulSoup(html, "html.parser")
    cards = soup.select(".product-card")
    products = []
    for card in cards:
        title_tag = card.select_one("a")
        price_tag = card.select_one(".price-wrapper")

        name = title_tag.select_one("h4").text.strip() if title_tag else None
        url = DOMAIN + title_tag["href"] if title_tag else None
        price = price_tag.text.strip() if price_tag else None

        products.append({
            "name": name,
            "price": price,
            "detail_url": url,
            "listing_page": page_num
        })
    return products

def get_total_results(soup):
    text = soup.get_text()
    match = re.search(r"([\d,]+)\s+results", text)
    return int(match.group(1).replace(",", "")) if match else None

all_products = []
page = 1
total_results = None

while True:
    resp = session.get(BASE_URL, params={"page": page})
    soup = BeautifulSoup(resp.text, "html.parser")

    if total_results is None:
        total_results = get_total_results(soup)
        print("Total results reported by site:", total_results)

    page_products = parse_listing_page(resp.text, page)

    if not page_products:          # empty page = stop
        print(f"Page {page} returned no products —> stopping.")
        break

    all_products.extend(page_products)
    print(f"Page {page}: collected {len(page_products)} (running total: {len(all_products)})")

    if total_results and len(all_products) >= total_results:
        print("Reached total_results count —> stopping.")
        break

    page += 1
    time.sleep(0.3)

print("Final count:", len(all_products))

Total results reported by site: 3000
Page 1: collected 32 (running total: 32)
Page 2: collected 32 (running total: 64)
Page 3: collected 32 (running total: 96)
Page 4: collected 32 (running total: 128)
Page 5: collected 32 (running total: 160)
Page 6: collected 32 (running total: 192)
Page 7: collected 32 (running total: 224)
Page 8: collected 32 (running total: 256)
Page 9: collected 32 (running total: 288)
Page 10: collected 32 (running total: 320)
Page 11: collected 32 (running total: 352)
Page 12: collected 32 (running total: 384)
Page 13: collected 32 (running total: 416)
Page 14: collected 32 (running total: 448)
Page 15: collected 32 (running total: 480)
Page 16: collected 32 (running total: 512)
Page 17: collected 32 (running total: 544)
Page 18: collected 32 (running total: 576)
Page 19: collected 32 (running total: 608)
Page 20: collected 32 (running total: 640)
Page 21: collected 32 (running total: 672)
Page 22: collected 32 (running total: 704)
Page 23: collected 32 (runnin

In [7]:
def parse_detail_page(html):
    soup = BeautifulSoup(html, "html.parser")
    avail_tag = soup.select_one(".availability")
    desc_tag = soup.select_one(".description")

    avail_text = avail_tag.text.strip().lower() if avail_tag else None
    stock_status = "out of stock" if avail_text and "out" in avail_text else "in stock" if avail_text else None

    description = desc_tag.text.strip() if desc_tag else None

    return {
        "stock_status": stock_status,
        "description": description
    }

def fetch_with_retry(url, session, max_retries=3, backoff=1.5):
    for attempt in range(1, max_retries + 1):
        try:
            resp = session.get(url, timeout=10)
            if resp.status_code == 200:
                return resp
            print(f"Attempt {attempt}: status {resp.status_code} for {url}")
        except requests.exceptions.RequestException as e:
            print(f"Attempt {attempt}: error {e} for {url}")
        time.sleep(backoff * attempt)   # wait longer each retry
    print(f"FAILED after {max_retries} attempts: {url}")
    return None

resp = fetch_with_retry("https://sandbox.oxylabs.io/products/1", session)
print(parse_detail_page(resp.text))

{'stock_status': 'in stock', 'description': 'As a young boy, Link is tricked by Ganondorf, the King of the Gerudo Thieves. The evil human uses Link to gain access to the Sacred Realm, where he places his tainted hands on Triforce and transforms the beautiful Hyrulean landscape into a barren wasteland. Link is determined to fix the problems he helped to create, so with the help of Rauru he travels through time gathering the powers of the Seven Sages.'}


In [8]:
import json

CHECKPOINT_EVERY = 200
CHECKPOINT_FILE = "checkpoint_static.json"

detailed_products = []
failed_urls = []

for i, product in enumerate(all_products):
    resp = fetch_with_retry(product["detail_url"], session)
    if resp is None:
        failed_urls.append(product["detail_url"])
        detail = {"stock_status": None, "description": None}
    else:
        detail = parse_detail_page(resp.text)

    merged = {**product, **detail}
    detailed_products.append(merged)

    if (i + 1) % CHECKPOINT_EVERY == 0:
        print(f"Progress: {i+1}/{len(all_products)}")
        with open(CHECKPOINT_FILE, "w") as f:
            json.dump(detailed_products, f)

    time.sleep(0.1)

print("Done. Total processed:", len(detailed_products))
print("Failed URLs:", len(failed_urls))

Progress: 200/3000
Progress: 400/3000
Progress: 600/3000
Progress: 800/3000
Progress: 1000/3000
Progress: 1200/3000
Progress: 1400/3000
Progress: 1600/3000
Progress: 1800/3000
Progress: 2000/3000
Progress: 2200/3000
Progress: 2400/3000
Progress: 2600/3000
Progress: 2800/3000
Progress: 3000/3000
Done. Total processed: 3000
Failed URLs: 0


In [9]:
test_slice = all_products[:5]
test_results = []
for p in test_slice:
    resp = fetch_with_retry(p["detail_url"], session)
    detail = parse_detail_page(resp.text)
    test_results.append({**p, **detail})

test_results

[{'name': 'The Legend of Zelda: Ocarina of Time',
  'price': '91,99 €',
  'detail_url': 'https://sandbox.oxylabs.io/products/1',
  'listing_page': 1,
  'stock_status': 'in stock',
  'description': 'As a young boy, Link is tricked by Ganondorf, the King of the Gerudo Thieves. The evil human uses Link to gain access to the Sacred Realm, where he places his tainted hands on Triforce and transforms the beautiful Hyrulean landscape into a barren wasteland. Link is determined to fix the problems he helped to create, so with the help of Rauru he travels through time gathering the powers of the Seven Sages.'},
 {'name': 'Super Mario Galaxy',
  'price': '91,99 €',
  'detail_url': 'https://sandbox.oxylabs.io/products/2',
  'listing_page': 1,
  'stock_status': 'out of stock',
  'description': "[Metacritic's 2007 Wii Game of the Year] The ultimate Nintendo hero is taking the ultimate step ... out into space. Join Mario as he ushers in a new era of video games, defying gravity across all the planet

In [10]:
df = pd.DataFrame(detailed_products)
print("Before dedup:", len(df))

df = df.drop_duplicates(subset="detail_url", keep="first")
print("After dedup:", len(df))

# reorder columns
df = df[["name", "price", "stock_status", "description", "detail_url", "listing_page"]]

filename = "23L_1008_versionA_static_products.csv"
df.to_csv(filename, index=False)
print(f"Saved {len(df)} rows to {filename}")

Before dedup: 3000
After dedup: 3000
Saved 3000 rows to 23L_1008_versionA_static_products.csv


In [11]:
print("=== Q1 Validation Statistics ===")
print("Total listing pages processed:", all_products[-1]["listing_page"])
print("Total records collected:", len(df))
print("Records with all required fields (no nulls):", df.dropna().shape[0])
print("Records missing stock_status or description:", df[df["stock_status"].isna() | df["description"].isna()].shape[0])
print("Failed detail-page fetches (after retries):", len(failed_urls))
print("In stock:", (df["stock_status"] == "in stock").sum())
print("Out of stock:", (df["stock_status"] == "out of stock").sum())

=== Q1 Validation Statistics ===
Total listing pages processed: 94
Total records collected: 3000
Records with all required fields (no nulls): 3000
Records missing stock_status or description: 0
Failed detail-page fetches (after retries): 0
In stock: 1500
Out of stock: 1500


QUESTION 2

In [19]:
!wget -q -O /tmp/chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y /tmp/chrome.deb -q
!google-chrome --version

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  at-spi2-common at-spi2-core gsettings-desktop-schemas libatk-bridge2.0-0t64
  libatk1.0-0t64 libatspi2.0-0t64 libxcomposite1 libxtst6 session-migration
The following NEW packages will be installed:
  at-spi2-common at-spi2-core google-chrome-stable gsettings-desktop-schemas
  libatk-bridge2.0-0t64 libatk1.0-0t64 libatspi2.0-0t64 libxcomposite1
  libxtst6 session-migration
0 upgraded, 10 newly installed, 0 to remove and 30 not upgraded.
Need to get 0 B/142 MB of archives.
After this operation, 456 MB of additional disk space will be used.
Get:1 /tmp/chrome.deb google-chrome-stable amd64 153.0.8010.36-1 [142 MB]
Selecting previously unselected package at-spi2-common.
(Reading database ... 123389 files and directories currently installed.)
Preparing to unpack .../0-at-spi2-common_2.52.0-1build1_all.deb ...
Unpacking at-spi2-common (2.52.0-1build1) ...
Se

In [20]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

options = Options()
options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
options.binary_location = "/usr/bin/google-chrome"

driver = webdriver.Chrome(options=options)
driver.get("https://www.scrapingcourse.com/infinite-scrolling")
time.sleep(2)
print(driver.title)

Infinite Scroll Challenge to Learn Web Scraping - ScrapingCourse.com


In [21]:
def parse_current_products(driver):
    cards = driver.find_elements(By.CSS_SELECTOR, ".product-item")
    products = []
    for card in cards:
        try:
            link = card.find_element(By.TAG_NAME, "a")
            url = link.get_attribute("href")
            name = card.find_element(By.CSS_SELECTOR, ".product-name").text.strip()
            price = card.find_element(By.CSS_SELECTOR, ".product-price").text.strip()
            image = card.find_element(By.CSS_SELECTOR, ".product-image").get_attribute("src")
            products.append({"name": name, "price": price, "detail_url": url, "image_url": image})
        except Exception as e:
            continue
    return products

# test products visible before any scrolling
initial_products = parse_current_products(driver)
print("Products visible before scrolling:", len(initial_products))
initial_products[:2]

Products visible before scrolling: 12


[{'name': 'Chaz Kangeroo Hoodie',
  'price': '$52',
  'detail_url': 'https://scrapingcourse.com/ecommerce/product/chaz-kangeroo-hoodie',
  'image_url': 'https://scrapingcourse.com/ecommerce/wp-content/uploads/2024/03/mh01-gray_main.jpg'},
 {'name': 'Teton Pullover Hoodie',
  'price': '$70',
  'detail_url': 'https://scrapingcourse.com/ecommerce/product/teton-pullover-hoodie',
  'image_url': 'https://scrapingcourse.com/ecommerce/wp-content/uploads/2024/03/mh02-black_main.jpg'}]

In [22]:
def scroll_and_collect(driver, max_scrolls=50, wait_timeout=10):
    seen_urls = {}   # detail_url -> batch number
    batch = 0

    # record initial (pre-scroll) products as batch 0
    current = parse_current_products(driver)
    for p in current:
        if p["detail_url"] not in seen_urls:
            seen_urls[p["detail_url"]] = {"batch": batch, **p}

    prev_count = len(current)
    print(f"Batch {batch}: {prev_count} products (initial)")

    for scroll_num in range(1, max_scrolls + 1):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")

        try:
            WebDriverWait(driver, wait_timeout).until(
                lambda d: len(d.find_elements(By.CSS_SELECTOR, ".product-item")) > prev_count
            )
        except:
            print(f"No new products after scroll {scroll_num} —> stopping.")
            break

        batch = scroll_num
        current = parse_current_products(driver)
        new_count = len(current)

        for p in current:
            if p["detail_url"] not in seen_urls:
                seen_urls[p["detail_url"]] = {"batch": batch, **p}

        print(f"Batch {batch}: {new_count} products total (+{new_count - prev_count} new)")
        prev_count = new_count

    return list(seen_urls.values())

all_scrolled_products = scroll_and_collect(driver)
print("\nTotal unique products collected:", len(all_scrolled_products))

Batch 0: 12 products (initial)
Batch 1: 24 products total (+12 new)
Batch 2: 36 products total (+12 new)
Batch 3: 48 products total (+12 new)
Batch 4: 60 products total (+12 new)
Batch 5: 72 products total (+12 new)
Batch 6: 84 products total (+12 new)
Batch 7: 96 products total (+12 new)
Batch 8: 108 products total (+12 new)
Batch 9: 120 products total (+12 new)
Batch 10: 132 products total (+12 new)
Batch 11: 144 products total (+12 new)
Batch 12: 156 products total (+12 new)
Batch 13: 168 products total (+12 new)
Batch 14: 180 products total (+12 new)
Batch 15: 187 products total (+7 new)
No new products after scroll 16 —> stopping.

Total unique products collected: 147


In [23]:
def parse_detail_page(html):
    soup = BeautifulSoup(html, "html.parser")
    sku_tag = soup.select_one(".sku")
    desc_tag = soup.select_one(".woocommerce-product-details__short-description p")

    sku = sku_tag.text.strip() if sku_tag else None
    short_description = desc_tag.text.strip() if desc_tag else None

    return {"sku": sku, "short_description": short_description}

# test with plain requests (separate session from Selenium driver)
detail_session = requests.Session()
headers = {"User-Agent": "Mozilla/5.0 (educational scraping assignment)"}

test_resp = detail_session.get("https://scrapingcourse.com/ecommerce/product/chaz-kangeroo-hoodie", headers=headers)
print(test_resp.status_code)
print(parse_detail_page(test_resp.text))

200
{'sku': 'MH01', 'short_description': 'This is a variable product called a Chaz Kangeroo Hoodie'}


In [24]:
final_dynamic_products = []
failed_detail_urls = []

for i, product in enumerate(all_scrolled_products):
    resp = fetch_with_retry(product["detail_url"], detail_session)
    if resp is None:
        failed_detail_urls.append(product["detail_url"])
        detail = {"sku": None, "short_description": None}
    else:
        detail = parse_detail_page(resp.text)

    merged = {**product, **detail}
    final_dynamic_products.append(merged)

    if (i + 1) % 25 == 0:
        print(f"Progress: {i+1}/{len(all_scrolled_products)}")

    time.sleep(0.2)

print("Done. Total:", len(final_dynamic_products))
print("Failed:", len(failed_detail_urls))

Progress: 25/147
Progress: 50/147
Progress: 75/147
Attempt 1: status 404 for https://scrapingcourse.com/ecommerce/product/juno-jacket
Attempt 2: status 404 for https://scrapingcourse.com/ecommerce/product/juno-jacket
Attempt 3: status 404 for https://scrapingcourse.com/ecommerce/product/juno-jacket
FAILED after 3 attempts: https://scrapingcourse.com/ecommerce/product/juno-jacket
Attempt 1: status 404 for https://scrapingcourse.com/ecommerce/product/inez-full-zip-jacket
Attempt 2: status 404 for https://scrapingcourse.com/ecommerce/product/inez-full-zip-jacket
Attempt 3: status 404 for https://scrapingcourse.com/ecommerce/product/inez-full-zip-jacket
FAILED after 3 attempts: https://scrapingcourse.com/ecommerce/product/inez-full-zip-jacket
Progress: 100/147
Attempt 1: status 404 for https://scrapingcourse.com/ecommerce/product/olivia-1/4-zip-light-jacket
Attempt 2: status 404 for https://scrapingcourse.com/ecommerce/product/olivia-1/4-zip-light-jacket
Attempt 3: status 404 for https://s

In [25]:
df2 = pd.DataFrame(final_dynamic_products)

# distinguish pre-scroll vs scroll-loaded
df2["loaded_via_scroll"] = df2["batch"] > 0

# reorder columns
df2 = df2[["name", "price", "image_url", "batch", "sku", "short_description", "detail_url", "loaded_via_scroll"]]
df2 = df2.rename(columns={"batch": "scroll_batch"})

filename2 = "23L_1008_versionA_dynamic_products.csv"
df2.to_csv(filename2, index=False)
print(f"Saved {len(df2)} rows to {filename2}")

df2.head(3)

Saved 147 rows to 23L_1008_versionA_dynamic_products.csv


,name,price,image_url,scroll_batch,sku,short_description,detail_url,loaded_via_scroll
0,Chaz Kangeroo Hoodie,$52,https://scrapingcourse.com/ecommerce/wp-conten...,0,MH01,This is a variable product called a Chaz Kange...,https://scrapingcourse.com/ecommerce/product/c...,False
1,Teton Pullover Hoodie,$70,https://scrapingcourse.com/ecommerce/wp-conten...,0,MH02,This is a variable product called a Teton Pull...,https://scrapingcourse.com/ecommerce/product/t...,False
2,Bruno Compete Hoodie,$63,https://scrapingcourse.com/ecommerce/wp-conten...,0,MH03,This is a variable product called a Bruno Comp...,https://scrapingcourse.com/ecommerce/product/b...,False


In [26]:
print("=== Q2 Validation Statistics ===")
print("Total scroll batches processed:", df2["scroll_batch"].max())
print("Total unique products collected:", len(df2))
print("Products present before scrolling (batch 0):", (df2["scroll_batch"] == 0).sum())
print("Products loaded via scrolling (batch > 0):", (df2["scroll_batch"] > 0).sum())
print("Records with all required fields (no nulls):", df2.dropna().shape[0])
print("Records missing sku or short_description:", df2[df2["sku"].isna() | df2["short_description"].isna()].shape[0])
print("Failed detail-page fetches (after retries):", len(failed_detail_urls))

=== Q2 Validation Statistics ===
Total scroll batches processed: 15
Total unique products collected: 147
Products present before scrolling (batch 0): 12
Products loaded via scrolling (batch > 0): 135
Records with all required fields (no nulls): 143
Records missing sku or short_description: 4
Failed detail-page fetches (after retries): 4


BONUS

In [27]:
!pip install playwright -q
!playwright install chromium
!playwright install-deps chromium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 9.4 MB/s eta 0:00:00
184.3 MiB [] 0% 373.9s184.3 MiB [] 0% 57.0s184.3 MiB [] 0% 34.7s184.3 MiB [] 0% 25.2s184.3 MiB [] 0% 22.7s184.3 MiB [] 0% 25.6s184.3 MiB [] 0% 19.1s184.3 MiB [] 0% 12.0s184.3 MiB [] 1% 9.1s184.3 MiB [] 2% 7.4s184.3 MiB [] 2% 6.9s184.3 MiB [] 2% 6.5s184.3 MiB [] 3% 6.4s184.3 MiB [] 3% 6.2s184.3 MiB [] 3% 6.0s184.3 MiB [] 4% 5.7s184.3 MiB [] 4% 5.6s184.3 MiB [] 4% 5.4s184.3 MiB [] 5% 5.4s184.3 MiB [] 5% 5.3s184.3 MiB [] 6% 5.2s184.3 MiB [] 6% 5.1s184.3 MiB [] 6% 5.7s184.3 MiB [] 7% 5.6s184.3 MiB [] 7% 5.7s184.3 MiB [] 7% 6.0s184.3 MiB [] 7% 6.1s184.3 MiB [] 8% 6.0s184.3 MiB [] 8% 5.9s184.3 MiB [] 9% 5.9s184.3 MiB [] 9% 5.7s184.3 MiB [] 10% 5.6s184.3 MiB [] 10% 5.4s184.3 MiB [] 11% 5.2s184.3 MiB [] 11% 5.0s184.3 MiB [] 12% 4.8s184.3 MiB [] 13% 4.7s184.3 MiB [] 13% 4.6s184.3 MiB [] 14% 4.6s184.3 MiB [] 14% 4.7s184.3 MiB [] 14% 4.6s184.3 MiB [] 15% 4.5s184.3 MiB [] 16% 4.5s184.3 MiB [] 16% 4.4s184.3 MiB [] 17% 4.3

In [28]:
from playwright.async_api import async_playwright
import asyncio

async def get_browser():
    playwright = await async_playwright().start()
    browser = await playwright.chromium.launch(headless=True)
    page = await browser.new_page(viewport={"width": 1920, "height": 1080})
    return playwright, browser, page

playwright, browser, page = await get_browser()
await page.goto("https://www.scrapingcourse.com/infinite-scrolling")
print(await page.title())

Infinite Scroll Challenge to Learn Web Scraping - ScrapingCourse.com


In [29]:
async def parse_current_products_pw(page):
    cards = await page.query_selector_all(".product-item")
    products = []
    for card in cards:
        try:
            link = await card.query_selector("a")
            url = await link.get_attribute("href")
            name_el = await card.query_selector(".product-name")
            name = (await name_el.inner_text()).strip()
            price_el = await card.query_selector(".product-price")
            price = (await price_el.inner_text()).strip()
            products.append({"name": name, "price": price, "detail_url": url})
        except Exception:
            continue
    return products

initial = await parse_current_products_pw(page)
print("Products before scrolling:", len(initial))

Products before scrolling: 12


In [30]:
async def scroll_and_collect_pw(page, max_scrolls=20, timeout_ms=8000):
    seen = {}
    batch = 0

    current = await parse_current_products_pw(page)
    for p in current:
        seen.setdefault(p["detail_url"], {"batch": batch, **p})

    prev_count = len(current)
    print(f"Batch {batch}: {prev_count} products (initial)")

    for scroll_num in range(1, max_scrolls + 1):
        await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")

        try:
            await page.wait_for_function(
                f"document.querySelectorAll('.product-item').length > {prev_count}",
                timeout=timeout_ms
            )
        except Exception:
            print(f"No new products after scroll {scroll_num} — stopping.")
            break

        batch = scroll_num
        current = await parse_current_products_pw(page)
        for p in current:
            seen.setdefault(p["detail_url"], {"batch": batch, **p})

        print(f"Batch {batch}: {len(current)} products total (+{len(current) - prev_count} new)")
        prev_count = len(current)

    return list(seen.values())

pw_products = await scroll_and_collect_pw(page)
print("\nTotal unique products:", len(pw_products))

Batch 0: 12 products (initial)
Batch 1: 24 products total (+12 new)
Batch 2: 36 products total (+12 new)
Batch 3: 48 products total (+12 new)
Batch 4: 60 products total (+12 new)
Batch 5: 72 products total (+12 new)
Batch 6: 84 products total (+12 new)
Batch 7: 96 products total (+12 new)
Batch 8: 108 products total (+12 new)
Batch 9: 120 products total (+12 new)
Batch 10: 132 products total (+12 new)
Batch 11: 144 products total (+12 new)
Batch 12: 156 products total (+12 new)
Batch 13: 168 products total (+12 new)
Batch 14: 180 products total (+12 new)
Batch 15: 187 products total (+7 new)
No new products after scroll 16 — stopping.

Total unique products: 147
